# TensorFlow: Basics to Advanced
### A Practical Notebook for Java Developers Transitioning to AI Engineering

---

**How to use this notebook**
- Run each cell top-to-bottom with `Shift+Enter`
- Every code cell has inline comments explaining what each line does
- Java analogies are included wherever TF concepts map to something you already know
- Cells marked `🟡 GPU RECOMMENDED` run fine on CPU but are faster on GPU — open in Google Colab for those

**What you will learn**
1. Tensors (TF's core data structure)
2. Tensor operations
3. Variables & automatic differentiation
4. Keras high-level API (Sequential, Functional, Subclassing)
5. Loss functions & optimizers
6. Training with `model.fit()` and manual training loops
7. `tf.data` pipelines
8. Callbacks, saving & loading models
9. CNNs, RNNs/LSTMs
10. Custom layers
11. Transfer learning
12. NLP / Text vectorization
13. Building a Transformer block from scratch
14. Production tips (`@tf.function`, SavedModel)


In [ ]:
# ── Install TensorFlow (run once) ──────────────────────────────────────────
# Uncomment the line below if TensorFlow is not installed yet
# !pip install tensorflow

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

# Check if a GPU is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU available: {gpus}")
else:
    print("No GPU found — running on CPU (fine for most chapters)")


---
## Chapter 1: Tensors — TensorFlow's Core Data Structure

A **tensor** is a multi-dimensional array of numbers — TensorFlow's equivalent of a Java `int[]`, `float[][]`, or `float[][][]`.

| Java | TensorFlow |
|---|---|
| `int x = 5` | `tf.constant(5)` — scalar (rank 0) |
| `int[] arr = {1,2,3}` | `tf.constant([1,2,3])` — vector (rank 1) |
| `int[][] mat = {{1,2},{3,4}}` | `tf.constant([[1,2],[3,4]])` — matrix (rank 2) |
| `int[][][]` | rank-3 tensor (e.g. a batch of images) |

**Key properties of a tensor:**
- `shape` → dimensions, like `.length` at each level
- `dtype` → data type (`float32`, `int32`, `bool`, etc.)
- `rank` → number of dimensions (`ndim`)

**Why tensors?** They live in GPU memory and support vectorized math (millions of operations in parallel). That's what makes deep learning fast.


In [ ]:
# ── Creating Tensors ──────────────────────────────────────────────────────

# Scalar — rank 0 (a single number)
scalar = tf.constant(42)
print("Scalar:", scalar)
print("  Shape:", scalar.shape)    # ()  — empty shape means scalar
print("  Dtype:", scalar.dtype)
print("  Rank:", scalar.ndim)

# Vector — rank 1 (1D array, like Java int[])
vector = tf.constant([1.0, 2.0, 3.0])
print("\nVector:", vector)
print("  Shape:", vector.shape)    # (3,)

# Matrix — rank 2 (2D array, like Java float[][])
matrix = tf.constant([[1, 2, 3],
                       [4, 5, 6]])
print("\nMatrix:", matrix)
print("  Shape:", matrix.shape)    # (2, 3) = 2 rows, 3 columns

# Rank-3 tensor (batch of images: batch_size x height x width)
tensor3d = tf.constant([[[1,2],[3,4]],
                         [[5,6],[7,8]]])
print("\n3D Tensor shape:", tensor3d.shape)  # (2, 2, 2)

# ── Useful factory functions ───────────────────────────────────────────────
zeros = tf.zeros([3, 4])          # 3x4 matrix of 0s — like new float[3][4]
ones  = tf.ones([2, 3])           # 2x3 matrix of 1s
eye   = tf.eye(3)                 # 3x3 identity matrix
rand  = tf.random.normal([2, 3])  # random values from normal distribution

print("\nZeros (3x4):\n", zeros.numpy())
print("\nIdentity (3x3):\n", eye.numpy())
print("\nRandom normal (2x3):\n", rand.numpy())


In [ ]:
# ── Tensor Properties & Conversion ────────────────────────────────────────

t = tf.constant([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

# Shape — like dimensions in Java arrays
print("Shape:", t.shape)          # TensorShape([2, 3])
print("Rows:", t.shape[0])        # 2
print("Cols:", t.shape[1])        # 3
print("Total elements:", tf.size(t).numpy())  # 6

# dtype — the data type stored
print("\nDtype:", t.dtype)        # float32 (default for decimals)

int_tensor = tf.constant([1, 2, 3])          # int32 by default
float_tensor = tf.cast(int_tensor, tf.float32) # cast — like (float) in Java
print("After cast:", float_tensor.dtype)

# Converting to/from NumPy
numpy_array = t.numpy()           # tensor → numpy array (CPU memory)
print("\nNumPy array:\n", numpy_array)
print("Type:", type(numpy_array))  # <class 'numpy.ndarray'>

# Create tensor from numpy
import numpy as np
np_arr = np.array([10, 20, 30])
from_np = tf.constant(np_arr)
print("\nFrom numpy:", from_np)

# ── Use Case ──────────────────────────────────────────────────────────────
# In neural networks, every layer's input/output is a tensor.
# A batch of 32 sentences, each tokenized to 128 tokens, each token = 768 dims:
# shape = (32, 128, 768) — exactly like the Q, K, V tensors in Transformers!
dummy_qkv = tf.random.normal([32, 128, 768])
print("\nQ/K/V tensor shape:", dummy_qkv.shape)  # (32, 128, 768)


---
## Chapter 2: Tensor Operations

TensorFlow overloads standard operators (`+`, `-`, `*`, `/`) for tensors.
Operations happen element-wise by default — like applying a `map()` across every element simultaneously on the GPU.

**Broadcasting** — when two tensors have different shapes, TF automatically expands the smaller one to match, similar to how Java's `Arrays.fill()` conceptually works on each row.


In [ ]:
# ── Element-wise Operations ───────────────────────────────────────────────

a = tf.constant([[1.0, 2.0],
                  [3.0, 4.0]])
b = tf.constant([[5.0, 6.0],
                  [7.0, 8.0]])

print("a + b:\n", (a + b).numpy())   # element-wise add
print("a * b:\n", (a * b).numpy())   # element-wise multiply (NOT matrix multiply)
print("a - b:\n", (a - b).numpy())
print("a / b:\n", (a / b).numpy())

# Math functions
print("\nSquare root:\n", tf.sqrt(a).numpy())
print("Exp:\n", tf.exp(a).numpy())
print("Log:\n", tf.math.log(a).numpy())

# ── Matrix Multiplication ─────────────────────────────────────────────────
# @ operator = matmul — the workhorse of neural networks
# This is how Q@K^T is computed in Transformer attention!

x = tf.constant([[1.0, 2.0, 3.0]])    # shape (1, 3)
W = tf.constant([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]])         # shape (3, 2)

result = x @ W                         # (1, 3) @ (3, 2) = (1, 2)
print("\nMatrix multiply (1,3) @ (3,2) =", result.numpy())

# Same as tf.linalg.matmul
result2 = tf.linalg.matmul(x, W)
print("Using tf.linalg.matmul:", result2.numpy())

# ── Reshape & Transpose ───────────────────────────────────────────────────
t = tf.constant([[1, 2, 3],
                  [4, 5, 6]])  # shape (2, 3)

reshaped = tf.reshape(t, [3, 2])     # like Java: rearrange elements into new shape
print("\nReshaped (3,2):\n", reshaped.numpy())

flat = tf.reshape(t, [-1])           # -1 means "figure it out" (flatten)
print("Flattened:", flat.numpy())

transposed = tf.transpose(t)         # flip rows ↔ columns: (2,3) → (3,2)
print("Transposed:\n", transposed.numpy())

# ── Reduction Operations ──────────────────────────────────────────────────
vals = tf.constant([[1.0, 2.0, 3.0],
                     [4.0, 5.0, 6.0]])

print("\nSum all:", tf.reduce_sum(vals).numpy())         # 21.0
print("Sum by row:", tf.reduce_sum(vals, axis=1).numpy()) # [6, 15]
print("Sum by col:", tf.reduce_sum(vals, axis=0).numpy()) # [5, 7, 9]
print("Mean:", tf.reduce_mean(vals).numpy())              # 3.5
print("Max:", tf.reduce_max(vals).numpy())                # 6.0

# ── Broadcasting ──────────────────────────────────────────────────────────
matrix = tf.constant([[1.0, 2.0, 3.0],   # shape (2, 3)
                        [4.0, 5.0, 6.0]])
bias   = tf.constant([10.0, 20.0, 30.0])  # shape (3,)

# TF automatically broadcasts bias across both rows — like adding a constant to each row
print("\nBroadcast (matrix + bias):\n", (matrix + bias).numpy())
# [[11, 22, 33], [14, 25, 36]]


---
## Chapter 3: Variables — Mutable Tensors (Model Weights)

`tf.constant` is immutable — like a Java `final` field. You cannot change its value.

`tf.Variable` is mutable — like a regular Java field. It CAN be changed.

**Why does this matter?** Model weights (W_Q, W_K, W_V, W_FF1, etc.) need to update during training. They are stored as `tf.Variable`. Constants (like positional encodings or fixed thresholds) use `tf.constant`.

```java
// Java analogy
final float[] CONSTANT = {1.0f, 2.0f};     // tf.constant — can't reassign
float[] weight = {0.1f, 0.3f};             // tf.Variable — updated each training step
```


In [ ]:
# ── tf.Variable ───────────────────────────────────────────────────────────

# Create a variable (like a model weight — starts random, gets updated during training)
w = tf.Variable([[0.1, 0.2],
                  [0.3, 0.4]], dtype=tf.float32)
b = tf.Variable([0.0, 0.0], dtype=tf.float32)  # bias vector

print("Weight variable:\n", w.numpy())
print("Bias variable:", b.numpy())
print("Is variable:", isinstance(w, tf.Variable))  # True

# ── Updating Variables ────────────────────────────────────────────────────
# .assign()     — set a new value (like w = newValue)
# .assign_add() — add to current value (like w += delta)
# .assign_sub() — subtract (like w -= delta)

b.assign([1.0, 2.0])          # replace bias
print("\nAfter assign:", b.numpy())

w.assign_add([[0.01, 0.01],   # add a small update (like one gradient step)
               [0.01, 0.01]])
print("Weight after assign_add:\n", w.numpy())

# ── Variables in neural network context ───────────────────────────────────
# This is EXACTLY how Keras stores W_Q, W_K, W_V under the hood:
d_model = 4
d_k     = 2

W_Q = tf.Variable(tf.random.normal([d_model, d_k]), name="W_Q")
W_K = tf.Variable(tf.random.normal([d_model, d_k]), name="W_K")
W_V = tf.Variable(tf.random.normal([d_model, d_k]), name="W_V")

print("\nW_Q (pretrained weight — d_model × d_k):")
print(W_Q.numpy())
print("Name:", W_Q.name)   # helps debugging — which layer owns this?

# ── trainable vs non-trainable ────────────────────────────────────────────
# trainable=True  (default) → gradient descent will update this weight
# trainable=False           → weight is frozen (used in transfer learning)

frozen_w = tf.Variable([1.0, 2.0], trainable=False, name="frozen_embed")
print("\nTrainable:", w.trainable)          # True
print("Non-trainable:", frozen_w.trainable) # False

# ── Use Case ──────────────────────────────────────────────────────────────
# When you call model.trainable_variables in Keras, you get a list of ALL
# tf.Variable objects that gradient descent is allowed to update.
# This is what gets passed to optimizer.apply_gradients() after each step.


---
## Chapter 4: Automatic Differentiation — `tf.GradientTape`

This is **the engine behind all neural network training**.

During training, we need to compute: *"how much should we change each weight to reduce the loss?"*
That answer is the **gradient** — the derivative of the loss with respect to each weight.

`tf.GradientTape` records every operation you do inside it, then lets you call `.gradient(loss, variables)` to compute gradients automatically. You never derive formulas by hand.

**Java analogy:** Imagine a very smart logging system that records every math operation you perform, then can replay it backwards to tell you the sensitivity of any output to any input.

```
Forward pass:  input → weights → output → loss       (recorded by tape)
Backward pass: loss.gradient(weights) → how to adjust weights  (tape replays backwards)
```


In [ ]:
# ── Basic Gradient Computation ────────────────────────────────────────────

# Simple example: y = x² → dy/dx = 2x
# At x=3: gradient should be 6.0

x = tf.Variable(3.0)  # must be a Variable (or watched constant) to track gradients

with tf.GradientTape() as tape:
    y = x ** 2         # tape records: "y was computed from x"

dy_dx = tape.gradient(y, x)   # compute dy/dx
print("y = x²  at x=3:")
print("  y value:", y.numpy())       # 9.0
print("  dy/dx:", dy_dx.numpy())     # 6.0  (= 2 * 3)

# ── Neural Network Forward Pass Example ───────────────────────────────────
# Simple linear layer: output = x @ W + b
# Loss = mean squared error between output and target

# Toy data
x_input = tf.constant([[1.0, 2.0, 3.0]])   # one sample, 3 features

# Weights (Variables — these get updated during training)
W = tf.Variable(tf.random.normal([3, 1]))  # 3 inputs → 1 output
b = tf.Variable(tf.zeros([1]))

target = tf.constant([[5.0]])  # what we want the output to be

with tf.GradientTape() as tape:
    # Forward pass: compute prediction
    output = x_input @ W + b             # linear layer
    loss   = tf.reduce_mean((output - target) ** 2)  # MSE loss

# Backward pass: compute gradients of loss with respect to W and b
grads = tape.gradient(loss, [W, b])

print("\nLinear layer example:")
print("  Output:", output.numpy())
print("  Loss:", loss.numpy())
print("  Gradient w.r.t W:", grads[0].numpy().T)  # how much to change W
print("  Gradient w.r.t b:", grads[1].numpy())    # how much to change b

# ── Manual Gradient Descent Step ──────────────────────────────────────────
learning_rate = 0.1

# Apply gradient: W = W - lr * gradient
W.assign_sub(learning_rate * grads[0])
b.assign_sub(learning_rate * grads[1])

# Recompute output after one update step
new_output = x_input @ W + b
new_loss   = tf.reduce_mean((new_output - target) ** 2)
print("\nAfter one gradient step:")
print("  New output:", new_output.numpy())
print("  New loss:", new_loss.numpy(), " (should be less than before)")

# ── Watching Constants ────────────────────────────────────────────────────
# By default, tape only tracks Variables. To track a constant, use tape.watch()
const_x = tf.constant(4.0)

with tf.GradientTape() as tape:
    tape.watch(const_x)          # explicitly watch this constant
    y = const_x ** 3             # y = x³ → dy/dx = 3x²

grad = tape.gradient(y, const_x)
print("\ny = x³ at x=4, dy/dx:", grad.numpy())  # 48.0 (= 3 * 16)

# ── Use Case ──────────────────────────────────────────────────────────────
# GradientTape is what Keras uses under the hood in model.fit().
# When you write a custom training loop (Chapter 10), you use GradientTape directly.
# Every weight update in GPT, BERT, LLaMA — all use this same mechanism.


---
## Chapter 5: Keras High-Level API — Sequential Model

**Keras** is TensorFlow's official high-level API. It lets you build models by stacking layers — like building with Lego blocks.

The **Sequential API** is the simplest: layers are stacked one after another, output of layer N feeds directly into layer N+1.

**Java analogy:** Sequential model = a Java method chain / pipeline:
```java
// Java stream pipeline
input.map(layer1).map(layer2).map(layer3).collect()

// Keras Sequential
model = Sequential([Dense(64), ReLU(), Dense(10), Softmax()])
```

**When to use Sequential:**
✅ Simple feedforward networks
✅ CNNs with one input & one output
❌ Multiple inputs/outputs
❌ Shared layers or skip connections


In [ ]:
# ── Sequential API: Linear Regression ─────────────────────────────────────
# Use case: predict house price from square footage

# Generate synthetic training data
np.random.seed(42)
X = np.random.uniform(500, 3000, size=(200, 1)).astype(np.float32)   # sq footage
y = (X * 150 + 50000 + np.random.normal(0, 20000, (200, 1))).astype(np.float32)  # price

# Normalize inputs (important for gradient-based training)
X_norm = (X - X.mean()) / X.std()
y_norm = (y - y.mean()) / y.std()

print("Input shape:", X_norm.shape)   # (200, 1)
print("Label shape:", y_norm.shape)   # (200, 1)

# ── Build the model ────────────────────────────────────────────────────────
model = tf.keras.Sequential([
    # Dense = fully-connected layer: output = input @ W + b
    # units=1 means one output neuron; input_shape=(1,) = one input feature
    tf.keras.layers.Dense(units=1, input_shape=(1,), name="linear_layer")
])

# Print model summary — shows layers, output shapes, parameter counts
model.summary()

# ── Compile the model ──────────────────────────────────────────────────────
# optimizer: how to update weights (Adam is the most common choice)
# loss: what to minimize (MSE for regression)
# metrics: what to print during training (not used for updating weights)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='mse',
    metrics=['mae']  # mean absolute error — easier to interpret than MSE
)

# ── Train the model ────────────────────────────────────────────────────────
# epochs: how many full passes over the training data
# validation_split: hold out 20% of data to measure generalization
history = model.fit(
    X_norm, y_norm,
    epochs=50,
    validation_split=0.2,
    verbose=0   # set to 1 to see epoch-by-epoch output
)
print("\nTraining complete!")
print("Final training loss:", round(history.history['loss'][-1], 4))
print("Final validation loss:", round(history.history['val_loss'][-1], 4))

# ── Evaluate & Predict ────────────────────────────────────────────────────
sample = np.array([[1500.0]], dtype=np.float32)
sample_norm = (sample - X.mean()) / X.std()
pred_norm = model.predict(sample_norm, verbose=0)
pred_price = pred_norm * y.std() + y.mean()
print(f"\nPredicted price for 1500 sqft: ${pred_price[0][0]:,.0f}")

# ── Plot training loss ─────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Sequential API: Multi-layer Neural Net — Binary Classification ──────────
# Use case: predict if a student passes (1) or fails (0) based on study hours + prev score

# Generate synthetic data
np.random.seed(0)
study   = np.random.uniform(0, 10, 300).astype(np.float32)
prev    = np.random.uniform(40, 100, 300).astype(np.float32)
passed  = ((study * 5 + prev) > 80).astype(np.float32)  # label

X_cls = np.column_stack([study, prev])
y_cls = passed

# Build multi-layer model
model_cls = tf.keras.Sequential([
    # Layer 1: 16 neurons, ReLU activation — extract basic patterns
    tf.keras.layers.Dense(16, activation='relu', input_shape=(2,), name="hidden1"),

    # Layer 2: 8 neurons — further abstraction
    tf.keras.layers.Dense(8, activation='relu', name="hidden2"),

    # Output layer: 1 neuron, sigmoid → outputs probability between 0 and 1
    # sigmoid(x) = 1 / (1 + e^-x) — squashes to [0, 1] for binary classification
    tf.keras.layers.Dense(1, activation='sigmoid', name="output")
])

model_cls.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # standard loss for binary classification
    metrics=['accuracy']
)

model_cls.summary()

history_cls = model_cls.fit(
    X_cls, y_cls,
    epochs=30,
    validation_split=0.2,
    verbose=0
)

_, acc = model_cls.evaluate(X_cls, y_cls, verbose=0)
print(f"\nAccuracy: {acc:.2%}")

# Predict on a new student: 8 hours study, 70 prev score
new_student = np.array([[8.0, 70.0]])
prob = model_cls.predict(new_student, verbose=0)[0][0]
print(f"New student pass probability: {prob:.2%} → {'PASS' if prob > 0.5 else 'FAIL'}")


---
## Chapter 6: Keras Functional API

The **Functional API** treats each layer as a callable function that takes and returns tensors.
This lets you build graphs with:
- **Multiple inputs** (text + image)
- **Multiple outputs** (sentiment + topic)
- **Skip connections** / residual connections (like in Transformers and ResNets)
- **Shared layers** (same layer applied to two different inputs)

**Java analogy:** Functional API = explicit DAG (Directed Acyclic Graph).
Like wiring together modules in a dependency injection framework — you control exactly which output feeds into which input.


In [ ]:
# ── Functional API: Multi-Input Model ────────────────────────────────────
# Use case: predict customer churn from BOTH numerical features AND text sentiment score

# ── Define inputs explicitly ───────────────────────────────────────────────
# Each Input() is an entry point — like method parameters
numeric_input = tf.keras.Input(shape=(5,), name="numeric_features")  # 5 numeric features
text_input    = tf.keras.Input(shape=(1,), name="sentiment_score")   # 1 sentiment score

# ── Process each input branch separately ──────────────────────────────────
# Branch 1: process numeric features
x1 = tf.keras.layers.Dense(32, activation='relu')(numeric_input)
x1 = tf.keras.layers.Dense(16, activation='relu')(x1)

# Branch 2: process text sentiment
x2 = tf.keras.layers.Dense(8, activation='relu')(text_input)

# ── Merge the two branches ─────────────────────────────────────────────────
# Concatenate: stick the two vectors side by side → (16 + 8 = 24) features
merged = tf.keras.layers.Concatenate()([x1, x2])

# ── Final output layers ───────────────────────────────────────────────────
x = tf.keras.layers.Dense(16, activation='relu')(merged)
output = tf.keras.layers.Dense(1, activation='sigmoid', name="churn_prob")(x)

# ── Build model by specifying inputs and outputs ───────────────────────────
model_func = tf.keras.Model(
    inputs=[numeric_input, text_input],
    outputs=output,
    name="churn_model"
)

model_func.summary()

# ── Compile and train ─────────────────────────────────────────────────────
model_func.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Synthetic data — two separate input arrays
np.random.seed(42)
numeric_data  = np.random.randn(500, 5).astype(np.float32)
sentiment_data = np.random.uniform(0, 1, (500, 1)).astype(np.float32)
labels = (numeric_data[:, 0] + sentiment_data[:, 0] > 0.5).astype(np.float32)

# Pass inputs as a LIST matching the order of model.inputs
model_func.fit(
    [numeric_data, sentiment_data], labels,
    epochs=10,
    validation_split=0.2,
    verbose=0
)
print("\nFunctional model training complete!")

# ── Skip Connection (Residual) — like in ResNets and Transformers ──────────
# Use case: the "Add & Norm" layers in every Transformer block are skip connections

inp = tf.keras.Input(shape=(64,), name="residual_input")
x   = tf.keras.layers.Dense(64, activation='relu')(inp)
x   = tf.keras.layers.Dense(64)(x)            # no activation before add

# Add original input back (skip connection / residual)
out = tf.keras.layers.Add()([inp, x])         # out = x + original_input
out = tf.keras.layers.LayerNormalization()(out)  # normalize (just like Transformers!)

residual_model = tf.keras.Model(inp, out, name="residual_block")
residual_model.summary()
print("\nResidual block model built — this is the exact pattern used in Transformers!")


---
## Chapter 7: Keras Model Subclassing — Full OOP Control

The **Subclassing API** lets you define a model as a Python class — exactly like extending an abstract class in Java.

```java
// Java analogy
abstract class Model {
    abstract Tensor forward(Tensor input);
}
class MyModel extends Model {
    Dense layer1 = new Dense(64);
    Dense layer2 = new Dense(10);
    Tensor forward(Tensor x) { return layer2.call(relu(layer1.call(x))); }
}
```

**When to use subclassing:**
✅ Complex architectures (Transformers, custom attention)
✅ Dynamic computation graphs (different logic per call based on input)
✅ Research / experimentation
❌ Simple feedforward models (use Sequential instead)


In [ ]:
# ── Subclassing: Build a Custom Model ────────────────────────────────────
# Use case: a classifier with a configurable number of hidden layers

class FlexibleClassifier(tf.keras.Model):
    """
    A feedforward classifier where you control the number of hidden layers.
    Java analogy: extends tf.keras.Model — must override __init__ and call()
    """

    def __init__(self, hidden_units_list, num_classes):
        # Call parent constructor — like super() in Java
        super(FlexibleClassifier, self).__init__()

        # Build hidden layers dynamically — like building a List<Layer>
        self.hidden_layers = []
        for units in hidden_units_list:
            self.hidden_layers.append(
                tf.keras.layers.Dense(units, activation='relu')
            )

        # Output layer
        activation = 'sigmoid' if num_classes == 1 else 'softmax'
        self.output_layer = tf.keras.layers.Dense(num_classes, activation=activation)

        # Dropout for regularization (randomly zeros some neurons during training)
        self.dropout = tf.keras.layers.Dropout(rate=0.3)

    def call(self, inputs, training=False):
        """
        Forward pass — like the main method of a Java class.
        training=True during model.fit(), False during model.evaluate()/predict()
        """
        x = inputs
        for layer in self.hidden_layers:
            x = layer(x)
            # Dropout only active during training (training=True)
            x = self.dropout(x, training=training)
        return self.output_layer(x)


# Instantiate — 3 hidden layers with 64, 32, 16 neurons; 3 output classes
model_sub = FlexibleClassifier(hidden_units_list=[64, 32, 16], num_classes=3)

# Compile
model_sub.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',  # for integer class labels (0, 1, 2)
    metrics=['accuracy']
)

# Synthetic multi-class data
X_mc = np.random.randn(600, 10).astype(np.float32)
y_mc = np.random.randint(0, 3, 600)  # labels: 0, 1, or 2

model_sub.fit(X_mc, y_mc, epochs=15, validation_split=0.2, verbose=0)

# Build model by passing a dummy input (needed to print summary for subclassed models)
model_sub.build(input_shape=(None, 10))
model_sub.summary()

# Access layers — like inspecting a Java object's fields
print("\nTrainable variable names:")
for v in model_sub.trainable_variables:
    print(f"  {v.name:40s} shape: {v.shape}")


---
## Chapter 8: Loss Functions & Optimizers

**Loss function** measures how wrong your model is. The optimizer's job is to minimize it.

| Task | Loss Function | Why |
|---|---|---|
| Regression | `mse` (Mean Squared Error) | Penalizes large errors strongly |
| Binary classification | `binary_crossentropy` | Measures probability distribution error |
| Multi-class classification | `categorical_crossentropy` | Same but for multiple classes |
| Multi-class (integer labels) | `sparse_categorical_crossentropy` | No need to one-hot encode labels |
| LLM / next-token prediction | `sparse_categorical_crossentropy` | Predicting 1 of 50,000 vocabulary words |

**Optimizer** decides *how* to apply the gradients:

| Optimizer | Description | When to use |
|---|---|---|
| `SGD` | Plain gradient descent, optionally with momentum | Simple baseline |
| `Adam` | Adaptive learning rate per weight | Default choice for most models |
| `AdamW` | Adam + weight decay regularization | Modern LLMs (GPT, LLaMA) |
| `RMSprop` | Adaptive lr, good for RNNs | Recurrent models |


In [ ]:
# ── Loss Functions ────────────────────────────────────────────────────────

# ── MSE (Mean Squared Error) — for regression ─────────────────────────────
mse_fn = tf.keras.losses.MeanSquaredError()
y_true_reg = tf.constant([3.0, -0.5, 2.0, 7.0])
y_pred_reg = tf.constant([2.5,  0.0, 2.0, 8.0])
print("MSE:", mse_fn(y_true_reg, y_pred_reg).numpy())

# ── Binary Crossentropy — for binary classification ────────────────────────
bce_fn = tf.keras.losses.BinaryCrossentropy()
y_true_bin = tf.constant([1.0, 0.0, 1.0, 0.0])
y_pred_bin = tf.constant([0.9, 0.1, 0.8, 0.2])   # model probabilities
print("Binary Crossentropy:", bce_fn(y_true_bin, y_pred_bin).numpy())

# ── Sparse Categorical Crossentropy — for multi-class, integer labels ─────
sce_fn = tf.keras.losses.SparseCategoricalCrossentropy()
y_true_mc = tf.constant([0, 1, 2])               # integer class labels
y_pred_mc = tf.constant([[0.8, 0.1, 0.1],        # model predicts class 0 — correct
                           [0.1, 0.7, 0.2],        # model predicts class 1 — correct
                           [0.3, 0.3, 0.4]])        # model predicts class 2 — correct
print("Sparse Cat. CE:", sce_fn(y_true_mc, y_pred_mc).numpy())

# ── Optimizers ────────────────────────────────────────────────────────────
# SGD — simplest optimizer
sgd = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)

# Adam — most common (adaptive per-weight learning rate)
adam = tf.keras.optimizers.Adam(
    learning_rate=0.001,  # default — usually works well
    beta_1=0.9,           # decay rate for first moment (gradient average)
    beta_2=0.999,         # decay rate for second moment (gradient variance)
    epsilon=1e-7          # prevents division by zero
)

# AdamW — Adam with weight decay (regularization to prevent overfitting)
# Used by modern LLMs
adamw = tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=0.01)

print("\nOptimizer configs:")
print("SGD lr:", sgd.learning_rate.numpy())
print("Adam lr:", adam.learning_rate.numpy())
print("AdamW wd:", adamw.weight_decay)

# ── Learning Rate Schedules ───────────────────────────────────────────────
# Modern LLMs use a warmup → decay schedule

# Cosine decay schedule
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.001,
    decay_steps=1000,    # reduce LR over 1000 steps
    alpha=0.0            # minimum lr fraction (0 = decay to 0)
)

# Plot how the learning rate changes over training steps
steps = np.arange(0, 1000)
lrs   = [lr_schedule(s).numpy() for s in steps]
plt.figure(figsize=(8,3))
plt.plot(steps, lrs)
plt.title("Cosine Learning Rate Decay Schedule")
plt.xlabel("Training step")
plt.ylabel("Learning rate")
plt.tight_layout()
plt.show()


---
## Chapter 9: Training with `model.fit()` — MNIST Digit Classification

**MNIST** is the "Hello World" of deep learning — 70,000 grayscale images of handwritten digits (0–9), each 28×28 pixels.

This chapter covers the **complete Keras training workflow**:
1. Load data
2. Preprocess
3. Build model
4. Compile (optimizer + loss + metrics)
5. Train (`model.fit()`)
6. Evaluate (`model.evaluate()`)
7. Predict (`model.predict()`)


In [ ]:
# ── Load and preprocess MNIST ──────────────────────────────────────────────
# TF includes MNIST built-in
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training images:", X_train.shape)   # (60000, 28, 28)
print("Test images:", X_test.shape)        # (10000, 28, 28)
print("Pixel range:", X_train.min(), "–", X_train.max())  # 0 – 255

# Normalize: scale pixels from [0, 255] → [0.0, 1.0]
# Neural networks converge much faster with normalized inputs
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

# Flatten 28×28 images to 784-element vectors for Dense layers
# (We'll use Conv2D for images in Chapter 14 — this is the simple version)
X_train_flat = X_train.reshape(-1, 784)   # (60000, 784)
X_test_flat  = X_test.reshape(-1, 784)    # (10000, 784)

# Show sample images
plt.figure(figsize=(10, 2))
for i in range(8):
    plt.subplot(1, 8, i+1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(str(y_train[i]))
    plt.axis('off')
plt.suptitle("Sample MNIST digits")
plt.tight_layout()
plt.show()

# ── Build model ────────────────────────────────────────────────────────────
mnist_model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dropout(0.2),    # randomly drop 20% of neurons each step
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')  # 10 classes (digits 0-9)
], name="mnist_classifier")

mnist_model.summary()

# ── Compile ───────────────────────────────────────────────────────────────
mnist_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── Train ────────────────────────────────────────────────────────────────
# batch_size=128: process 128 samples at once per gradient step
# epochs=10: go through all 60,000 training images 10 times
history_mnist = mnist_model.fit(
    X_train_flat, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,  # use 6,000 images for validation
    verbose=1
)

# ── Evaluate ──────────────────────────────────────────────────────────────
test_loss, test_acc = mnist_model.evaluate(X_test_flat, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc:.2%}")
print(f"Test Loss:     {test_loss:.4f}")

# ── Predict ───────────────────────────────────────────────────────────────
sample_imgs = X_test_flat[:8]
probs = mnist_model.predict(sample_imgs, verbose=0)   # (8, 10) — prob for each digit
predicted_labels = np.argmax(probs, axis=1)
true_labels = y_test[:8]

print("\nPredicted:", predicted_labels)
print("Actual:   ", true_labels)

plt.figure(figsize=(10, 2))
for i in range(8):
    plt.subplot(1, 8, i+1)
    plt.imshow(X_test[i], cmap='gray')
    color = 'green' if predicted_labels[i] == true_labels[i] else 'red'
    plt.title(str(predicted_labels[i]), color=color)
    plt.axis('off')
plt.suptitle("Predictions (green=correct, red=wrong)")
plt.tight_layout()
plt.show()


---
## Chapter 10: Manual Training Loop (Low-Level)

`model.fit()` is convenient but hides what's happening. A **manual training loop** gives you full control:
- Custom loss terms
- Gradient clipping
- Multiple optimizers (e.g. GANs)
- Per-batch logging

This is also exactly what research papers implement. Understanding this = understanding how LLMs are trained.

**Every training step:**
1. Forward pass inside `GradientTape`
2. Compute loss
3. Compute gradients with `tape.gradient(loss, model.trainable_variables)`
4. Apply gradients: `optimizer.apply_gradients(zip(grads, model.trainable_variables))`


In [ ]:
# ── Manual Training Loop ─────────────────────────────────────────────────
# Same MNIST problem — but now we control every step

# Reuse the MNIST data from Chapter 9
# Build a fresh model
manual_model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10)  # no activation — we'll use from_logits=True in loss
])

optimizer = tf.keras.optimizers.Adam(0.001)
loss_fn   = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ── Build tf.data dataset for efficient batching ───────────────────────────
BATCH_SIZE = 128
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_flat, y_train))
train_dataset = train_dataset.shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ── Define one training step as a function ────────────────────────────────
# @tf.function compiles this to a static computation graph → faster (Chapter 21)
@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        logits = manual_model(x_batch, training=True)  # forward pass
        loss   = loss_fn(y_batch, logits)               # compute loss

    # Compute gradients of loss w.r.t. ALL trainable weights
    grads = tape.gradient(loss, manual_model.trainable_variables)

    # Gradient clipping — prevents gradients from becoming too large (important for RNNs)
    grads, _ = tf.clip_by_global_norm(grads, clip_norm=1.0)

    # Update weights: w = w - lr * gradient
    optimizer.apply_gradients(zip(grads, manual_model.trainable_variables))
    return loss

# ── Define validation step ─────────────────────────────────────────────────
@tf.function
def val_step(x_batch, y_batch):
    logits = manual_model(x_batch, training=False)  # no dropout during eval
    loss   = loss_fn(y_batch, logits)
    preds  = tf.argmax(logits, axis=1)
    acc    = tf.reduce_mean(tf.cast(tf.equal(preds, tf.cast(y_batch, tf.int64)), tf.float32))
    return loss, acc

# ── Training loop ─────────────────────────────────────────────────────────
EPOCHS = 5
val_data = tf.data.Dataset.from_tensor_slices((X_test_flat, y_test)).batch(256)

for epoch in range(EPOCHS):
    # Train
    train_losses = []
    for x_batch, y_batch in train_dataset:
        loss = train_step(x_batch, y_batch)
        train_losses.append(loss.numpy())

    # Validate
    val_losses, val_accs = [], []
    for x_val, y_val in val_data:
        vl, va = val_step(x_val, y_val)
        val_losses.append(vl.numpy())
        val_accs.append(va.numpy())

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {np.mean(train_losses):.4f} | "
          f"Val Loss: {np.mean(val_losses):.4f} | "
          f"Val Acc: {np.mean(val_accs):.2%}")

print("\nManual training loop complete!")
print("This is exactly what happens inside model.fit() — now you see the internals.")


---
## Chapter 11: `tf.data` — Efficient Data Pipelines

In production, data loading is often the bottleneck. `tf.data` lets you build fast, efficient pipelines with:
- **Parallelism:** load & preprocess while GPU trains on previous batch
- **Streaming:** handle datasets too large for RAM
- **Augmentation:** apply random transforms on-the-fly

**Java analogy:** `tf.data.Dataset` is like a Java `Stream` — lazy, composable, parallelizable.


In [ ]:
# ── Creating Datasets ─────────────────────────────────────────────────────

# ── From numpy arrays (most common) ────────────────────────────────────────
X_demo = np.arange(100).reshape(100, 1).astype(np.float32)
y_demo = (X_demo * 2 + 1).reshape(-1)

dataset = tf.data.Dataset.from_tensor_slices((X_demo, y_demo))
print("Dataset:", dataset)
print("Element spec:", dataset.element_spec)

# Peek at first 3 samples
for x, y in dataset.take(3):
    print(f"  x={x.numpy()[0]:.0f}, y={y.numpy():.0f}")

# ── Pipeline operations ────────────────────────────────────────────────────
pipeline = (
    tf.data.Dataset.from_tensor_slices((X_demo, y_demo))
    .shuffle(buffer_size=100, seed=42)   # shuffle (important: buffer_size ≥ dataset size)
    .batch(16)                           # group into batches of 16
    .prefetch(tf.data.AUTOTUNE)          # load next batch while GPU trains on current
)

print("\nPipeline:", pipeline)
print("Output shapes:", pipeline.element_spec)

# ── map() — transform each element ────────────────────────────────────────
# Java analogy: stream.map()

# Normalize features on-the-fly
def normalize(x, y):
    x = (x - 50.0) / 28.9  # standardize
    return x, y

mapped_ds = (
    tf.data.Dataset.from_tensor_slices((X_demo, y_demo))
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)  # parallel transform
    .batch(16)
    .prefetch(tf.data.AUTOTUNE)
)

# ── filter() — keep only certain samples ──────────────────────────────────
# Java analogy: stream.filter()
even_ds = (
    tf.data.Dataset.from_tensor_slices((X_demo, y_demo))
    .filter(lambda x, y: x[0] % 2 == 0)  # keep only even x values
)
print("\nFiltered samples (first 5):")
for x, y in even_ds.take(5):
    print(f"  x={x.numpy()[0]:.0f}")

# ── From a generator — useful for large/custom datasets ───────────────────
def data_generator():
    """Generates samples on demand — never loads all data into memory"""
    for i in range(50):
        x = np.random.randn(4).astype(np.float32)
        y = np.random.randint(0, 3)
        yield x, y

gen_ds = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=(
        tf.TensorSpec(shape=(4,), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
)
print("\nGenerator dataset (first 3):")
for x, y in gen_ds.take(3):
    print(f"  x={x.numpy()}, y={y.numpy()}")

# ── Image augmentation example (data augmentation during training) ─────────
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),      # randomly flip image
    tf.keras.layers.RandomRotation(0.1),           # rotate ±10%
    tf.keras.layers.RandomZoom(0.1),               # zoom in/out ±10%
])
print("\nImage augmentation pipeline defined (apply .map(augment) to image dataset)")
print("Use case: prevents overfitting — model sees slightly different version each epoch")


---
## Chapter 12: Callbacks — Hooks Into the Training Loop

**Callbacks** are objects that get called at specific points during training:
- end of each epoch
- end of each batch
- when training starts/ends

**Java analogy:** Callbacks are like `EventListener` implementations — you register them and the training loop calls them at the right time.

Common callbacks:
| Callback | Purpose |
|---|---|
| `EarlyStopping` | Stop if validation loss stops improving (prevents overfitting & wasted time) |
| `ModelCheckpoint` | Save the best model weights to disk automatically |
| `ReduceLROnPlateau` | Cut learning rate when loss plateaus |
| `TensorBoard` | Write training metrics to TensorBoard for visualization |
| Custom callback | Any logic you want (e.g. send Slack notification when training ends) |


In [ ]:
# ── EarlyStopping ─────────────────────────────────────────────────────────
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',   # watch validation loss
    patience=3,           # stop if no improvement for 3 consecutive epochs
    restore_best_weights=True,  # roll back to the best epoch's weights
    verbose=1
)

# ── ModelCheckpoint ────────────────────────────────────────────────────────
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_model.keras',  # save to this file
    monitor='val_accuracy',
    save_best_only=True,          # only save when val_accuracy improves
    verbose=1
)

# ── ReduceLROnPlateau ──────────────────────────────────────────────────────
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,        # multiply lr by 0.5 when triggered
    patience=2,        # trigger if no improvement for 2 epochs
    min_lr=1e-6,       # never go below this lr
    verbose=1
)

# ── Custom Callback ────────────────────────────────────────────────────────
class TrainingLogger(tf.keras.callbacks.Callback):
    """
    Custom callback: prints a summary at the end of each epoch.
    Java analogy: implements EventListener<EpochEndEvent>
    """
    def on_epoch_end(self, epoch, logs=None):
        lr = float(self.model.optimizer.learning_rate)
        print(f"  [Logger] Epoch {epoch+1}: "
              f"loss={logs['loss']:.4f}, "
              f"val_acc={logs.get('val_accuracy', 0):.2%}, "
              f"lr={lr:.6f}")

    def on_train_end(self, logs=None):
        print("  [Logger] Training finished!")

# ── Train MNIST model with all callbacks ──────────────────────────────────
cb_model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])
cb_model.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

print("Training with callbacks (EarlyStopping will kick in if val_loss doesn't improve):\n")
cb_model.fit(
    X_train_flat, y_train,
    epochs=50,           # max 50 epochs, but EarlyStopping may stop earlier
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr, TrainingLogger()],
    verbose=0            # suppress default output — TrainingLogger handles it
)


---
## Chapter 13: Saving & Loading Models

Once trained, you need to persist models. TF offers two main formats:

| Format | Use case |
|---|---|
| **Keras native** (`.keras`) | Save/load within Python for continued training |
| **SavedModel** (directory) | Production deployment (TF Serving, TFLite, TF.js) |
| **Weights only** (`.weights.h5`) | Save just the weights, recreate architecture in code |

**Java analogy:**
- `.keras` = Java object serialization (for development)
- `SavedModel` = compiled JAR for deployment (frozen, language-agnostic)


In [ ]:
# ── Save entire model (architecture + weights + optimizer state) ──────────
mnist_model.save('mnist_model.keras')
print("Model saved as mnist_model.keras")

# ── Load model ────────────────────────────────────────────────────────────
loaded_model = tf.keras.models.load_model('mnist_model.keras')
loaded_model.summary()

# Verify loaded model gives same predictions
orig_pred  = np.argmax(mnist_model.predict(X_test_flat[:5], verbose=0), axis=1)
load_pred  = np.argmax(loaded_model.predict(X_test_flat[:5], verbose=0), axis=1)
print("Original predictions:", orig_pred)
print("Loaded predictions:  ", load_pred)
assert np.array_equal(orig_pred, load_pred), "Predictions don't match!"
print("✓ Loaded model gives identical predictions")

# ── Save as SavedModel format (for production deployment) ─────────────────
mnist_model.export('mnist_saved_model')  # creates a directory
print("\nSaved as SavedModel directory: mnist_saved_model/")

# Load SavedModel
import os
if os.path.exists('mnist_saved_model'):
    # For inference, load with tf.saved_model
    infer_fn = tf.saved_model.load('mnist_saved_model')
    print("SavedModel loaded for inference")

# ── Save/load weights only ────────────────────────────────────────────────
mnist_model.save_weights('mnist_weights.weights.h5')
print("\nWeights saved to mnist_weights.weights.h5")

# To reload weights: recreate same architecture first, then load weights
new_model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])
new_model.load_weights('mnist_weights.weights.h5')
print("Weights loaded into fresh model")

# ── Use Case ──────────────────────────────────────────────────────────────
# ModelCheckpoint (Chapter 12) uses save_weights() internally.
# In production: SavedModel format is deployed to TF Serving — a REST API
# server that loads your model and serves predictions at low latency.
print("\nUse Case: Deploy SavedModel to TensorFlow Serving REST API")
print("  POST /v1/models/mnist_model:predict  →  {predictions: [3]}")


---
## Chapter 14: Convolutional Neural Networks (CNNs)
### 🟡 GPU RECOMMENDED — Open in Google Colab for faster training

CNNs are the standard architecture for image data. Instead of connecting every pixel to every neuron (which would be millions of parameters), CNNs use **sliding filters** that detect local patterns (edges, shapes, textures).

**Key layers:**
- `Conv2D` — applies a set of filters over the image (detects local patterns)
- `MaxPooling2D` — downsamples spatial dimensions (reduces computation)
- `Flatten` / `GlobalAveragePooling2D` — converts 2D feature maps to 1D vector
- `Dense` — final classification layers

**Java analogy:** A convolutional filter is like a method that slides over a 2D array, applying the same computation to every overlapping window.

**Use cases:** Image classification, object detection, face recognition, medical imaging.


In [ ]:
# ── CNN for MNIST Image Classification ────────────────────────────────────
# 🟡 GPU RECOMMENDED — runs in ~2 min on GPU, ~15 min on CPU

# Reload MNIST with correct shape for Conv2D: (samples, height, width, channels)
(X_train_img, y_train), (X_test_img, y_test) = tf.keras.datasets.mnist.load_data()

# Add channel dimension: (60000, 28, 28) → (60000, 28, 28, 1)  — grayscale = 1 channel
X_train_img = X_train_img[..., np.newaxis].astype('float32') / 255.0
X_test_img  = X_test_img[..., np.newaxis].astype('float32') / 255.0

print("CNN input shape:", X_train_img.shape)  # (60000, 28, 28, 1)

# ── Build CNN ──────────────────────────────────────────────────────────────
cnn_model = tf.keras.Sequential([
    # Conv block 1: detect edges and simple patterns
    # filters=32: learn 32 different filters
    # kernel_size=3: each filter is 3×3 pixels
    # padding='same': output same spatial size as input
    tf.keras.layers.Conv2D(32, kernel_size=3, activation='relu',
                            padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D(pool_size=2),  # 28×28 → 14×14

    # Conv block 2: detect more complex patterns from previous features
    tf.keras.layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(pool_size=2),  # 14×14 → 7×7

    # Conv block 3: high-level feature detection
    tf.keras.layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'),

    # Flatten: convert 7×7×64 feature map → 3136 vector
    tf.keras.layers.Flatten(),

    # Classification head
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
], name="cnn_mnist")

cnn_model.summary()  # note: far fewer params than the flat Dense model, but more powerful

# ── Compile & Train ────────────────────────────────────────────────────────
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.fit(
    X_train_img, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
    verbose=1
)

test_loss, test_acc = cnn_model.evaluate(X_test_img, y_test, verbose=0)
print(f"\nCNN Test Accuracy: {test_acc:.2%}")
print("Dense model was ~97% — CNN should reach ~99%+ showing the benefit of convolutions")

# ── Visualize what the CNN learned ────────────────────────────────────────
# Show the 32 filters from the first Conv2D layer
first_conv_weights = cnn_model.layers[0].get_weights()[0]  # shape: (3, 3, 1, 32)
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(first_conv_weights[:, :, 0, i], cmap='gray')
    ax.axis('off')
plt.suptitle("32 learned Conv2D filters (3×3 each) — these detect edges, curves, etc.")
plt.tight_layout()
plt.show()


---
## Chapter 15: Recurrent Neural Networks — RNNs & LSTMs

RNNs process **sequences** — text, time series, audio. Each step reads the current input AND a hidden state carrying memory from all previous steps.

**LSTM** (Long Short-Term Memory) solves the vanishing gradient problem of plain RNNs by adding gating mechanisms that control what to remember and forget.

**Java analogy:** An LSTM is like a stateful iterator — it processes one element at a time but carries a "memory object" that persists and updates between iterations.

**Note:** For modern NLP, Transformers (Chapter 17+) outperform LSTMs. But LSTMs are still widely used for time series, IoT sensor data, and financial prediction where sequence length is moderate.

**Use cases:** Sentiment analysis, time series forecasting, text generation, speech recognition.


In [ ]:
# ── LSTM for Text Sentiment Analysis ──────────────────────────────────────
# Use case: classify IMDB movie reviews as positive (1) or negative (0)

# Load IMDB dataset (50,000 reviews, already tokenized as integer sequences)
MAX_FEATURES = 10000  # vocabulary size (top 10k most common words)
MAX_LEN      = 200    # max review length (pad/truncate to 200 tokens)

(X_train_imdb, y_train_imdb), (X_test_imdb, y_test_imdb) =     tf.keras.datasets.imdb.load_data(num_words=MAX_FEATURES)

print("IMDB loaded:")
print("  Train samples:", len(X_train_imdb))
print("  Sample review (first 10 token IDs):", X_train_imdb[0][:10])
print("  Label (0=neg, 1=pos):", y_train_imdb[0])

# Pad/truncate all reviews to same length (required for batching)
# Like padding Java byte arrays to a fixed size
X_train_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_train_imdb, maxlen=MAX_LEN, padding='post', truncating='post'
)
X_test_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_test_imdb, maxlen=MAX_LEN, padding='post', truncating='post'
)
print("\nAfter padding:", X_train_pad.shape)  # (25000, 200)

# ── Build LSTM model ────────────────────────────────────────────────────────
lstm_model = tf.keras.Sequential([
    # Embedding: convert token IDs → dense vectors (same concept as W_e in Transformers)
    # input_dim=10000: vocab size  |  output_dim=64: embedding dimensions
    tf.keras.layers.Embedding(input_dim=MAX_FEATURES, output_dim=64, name="embedding"),

    # Bidirectional LSTM: process sequence in BOTH directions (forward + backward)
    # return_sequences=False: only return the final hidden state (not all steps)
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2)
    ),

    # Classification head
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # binary: pos/neg
], name="lstm_sentiment")

lstm_model.summary()

# ── Compile & Train ────────────────────────────────────────────────────────
lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_history = lstm_model.fit(
    X_train_pad, y_train_imdb,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

test_loss, test_acc = lstm_model.evaluate(X_test_pad, y_test_imdb, verbose=0)
print(f"\nLSTM Test Accuracy: {test_acc:.2%}")

# ── Predict on custom review ───────────────────────────────────────────────
# Get the word index to tokenize custom text
word_index = tf.keras.datasets.imdb.get_word_index()

def encode_review(text):
    tokens = [word_index.get(w.lower(), 2) for w in text.split()]
    tokens = [min(t, MAX_FEATURES - 1) for t in tokens]  # clamp to vocab size
    padded = tf.keras.preprocessing.sequence.pad_sequences([tokens], maxlen=MAX_LEN)
    return padded

review1 = "this movie was absolutely fantastic the acting was brilliant"
review2 = "terrible waste of time boring and predictable plot"

for review in [review1, review2]:
    encoded = encode_review(review)
    prob    = lstm_model.predict(encoded, verbose=0)[0][0]
    print(f"\nReview: '{review[:50]}...'")
    print(f"  Sentiment: {'POSITIVE' if prob > 0.5 else 'NEGATIVE'} ({prob:.2%})")


---
## Chapter 16: Custom Layers — Building Your Own

When built-in Keras layers don't fit, you create your own by subclassing `tf.keras.layers.Layer`.

**Required methods:**
- `__init__`: define layer config, call `super().__init__()`
- `build(input_shape)`: create weights (called once on first forward pass)
- `call(inputs)`: forward pass logic

**Java analogy:** Like implementing an interface with a required `forward(Tensor input)` method. `build()` is called lazily once — like a lazy initializer.


In [ ]:
# ── Custom Layer: Linear Layer from Scratch ───────────────────────────────
# Replicates what tf.keras.layers.Dense does internally

class MyLinear(tf.keras.layers.Layer):
    """
    Custom fully-connected layer: output = input @ W + b
    Equivalent to tf.keras.layers.Dense — built from scratch for learning.
    """

    def __init__(self, units, activation=None, **kwargs):
        super(MyLinear, self).__init__(**kwargs)
        self.units      = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        # build() is called once — creates the weight variables
        # input_shape[-1] = number of input features (inferred automatically)
        self.W = self.add_weight(
            name='kernel',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',  # Xavier init — good default for dense layers
            trainable=True
        )
        self.b = self.add_weight(
            name='bias',
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        super(MyLinear, self).build(input_shape)  # marks layer as built

    def call(self, inputs):
        output = tf.linalg.matmul(inputs, self.W) + self.b
        if self.activation:
            output = self.activation(output)
        return output

    def get_config(self):
        # Needed to serialize/load the layer
        config = super().get_config()
        config.update({'units': self.units, 'activation': self.activation})
        return config


# Test custom layer
x = tf.constant([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
my_layer = MyLinear(units=4, activation='relu')
output = my_layer(x)
print("MyLinear output shape:", output.shape)  # (2, 4)
print("Weights:", my_layer.weights[0].shape)   # (3, 4)

# ── Custom Layer: Scaled Dot-Product Attention ─────────────────────────────
# The core of every Transformer — implements the attention formula:
# Attention(Q, K, V) = softmax(Q @ K.T / sqrt(d_k)) @ V

class ScaledDotProductAttention(tf.keras.layers.Layer):
    """
    Implements the attention mechanism at the heart of every Transformer.
    This is W_Q, W_K, W_V in action.
    """

    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

        # Three learned weight matrices — W_Q, W_K, W_V
        # Each is a Dense layer (without activation)
        self.W_Q = tf.keras.layers.Dense(d_model, use_bias=False, name="W_Q")
        self.W_K = tf.keras.layers.Dense(d_model, use_bias=False, name="W_K")
        self.W_V = tf.keras.layers.Dense(d_model, use_bias=False, name="W_V")

    def call(self, query, key, value, mask=None):
        # Project inputs through learned weight matrices
        Q = self.W_Q(query)    # (batch, seq_len, d_model)
        K = self.W_K(key)
        V = self.W_V(value)

        # Scale factor: 1 / sqrt(d_model)
        scale = tf.math.sqrt(tf.cast(self.d_model, tf.float32))

        # Attention scores: Q @ K^T / sqrt(d_k)
        scores = tf.linalg.matmul(Q, K, transpose_b=True) / scale  # (batch, seq, seq)

        # Optional mask (for causal/decoder attention)
        if mask is not None:
            scores += (mask * -1e9)  # add -inf to masked positions → softmax gives 0

        # Attention weights (probabilities per position)
        weights = tf.nn.softmax(scores, axis=-1)  # (batch, seq, seq)

        # Weighted sum of values
        output = tf.linalg.matmul(weights, V)  # (batch, seq, d_model)
        return output, weights


# Test attention layer
batch, seq_len, d_model = 2, 5, 16
x_attn = tf.random.normal([batch, seq_len, d_model])

attn_layer = ScaledDotProductAttention(d_model=d_model)
output, weights = attn_layer(x_attn, x_attn, x_attn)

print("\nAttention test:")
print("  Input shape:", x_attn.shape)
print("  Output shape:", output.shape)   # same as input — attention is shape-preserving
print("  Attention weights shape:", weights.shape)  # (2, 5, 5) — each token attends to all
print("  Weights sum per row:", weights[0, 0].numpy().sum())  # should be 1.0 (softmax)


---
## Chapter 17: Transfer Learning
### 🟡 GPU RECOMMENDED — Open in Google Colab

**Transfer learning** = take a model pre-trained on a large dataset, reuse its learned representations, and fine-tune on your smaller ta dataset.

Why it works: the first layers of any image model learn universal features (edges, textures, shapes) that transfer to any visual task. You only retrain the last few layers.

**Java analogy:** Like inheriting a base class that already has all the hard algorithmic work done — you just override the final output method for your specific use case.

**Strategy:**
1. Load pre-trained model (e.g. MobileNetV2, EfficientNet, BERT)
2. Freeze base layers (`trainable = False`) — their weights don't change
3. Add your custom classification head on top
4. Train only the head first (fast, small dataset)
5. Optionally "unfreeze" top base layers for fine-tuning (more training, better accuracy)


In [ ]:
# ── Transfer Learning with MobileNetV2 ────────────────────────────────────
# 🟡 GPU RECOMMENDED
# Use case: classify cats vs dogs (binary) using a model pre-trained on ImageNet

# Load CIFAR-10 as a proxy (real cats/dogs would need larger images)
(X_train_c, y_train_c), (X_test_c, y_test_c) = tf.keras.datasets.cifar10.load_data()

# CIFAR-10 has 10 classes — we use only cats(3) and dogs(5) for binary task
cat_dog_mask_train = (y_train_c[:, 0] == 3) | (y_train_c[:, 0] == 5)
cat_dog_mask_test  = (y_test_c[:, 0] == 3)  | (y_test_c[:, 0] == 5)

X_train_cd = X_train_c[cat_dog_mask_train].astype('float32') / 255.0
y_train_cd = (y_train_c[cat_dog_mask_train, 0] == 5).astype('float32')  # dog=1, cat=0
X_test_cd  = X_test_c[cat_dog_mask_test].astype('float32') / 255.0
y_test_cd  = (y_test_c[cat_dog_mask_test, 0] == 5).astype('float32')

print("Cat/Dog dataset:")
print("  Train:", X_train_cd.shape, "| Labels:", y_train_cd.shape)
print("  Test:", X_test_cd.shape)

# MobileNetV2 expects 96×96 or 224×224 images; CIFAR is 32×32 — resize
X_train_resized = tf.image.resize(X_train_cd, [96, 96])
X_test_resized  = tf.image.resize(X_test_cd,  [96, 96])

# ── Load pre-trained base model ────────────────────────────────────────────
# include_top=False: remove the ImageNet classifier head (1000 classes)
# We'll add our own 2-class head
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'         # pre-trained on 1.2M ImageNet images
)

print("\nBase model layers:", len(base_model.layers))
print("Base model output shape:", base_model.output_shape)  # (None, 3, 3, 1280)

# ── Freeze base model ──────────────────────────────────────────────────────
# trainable=False means gradient descent will NOT update these weights
base_model.trainable = False
frozen_count = sum(not v.trainable for v in base_model.trainable_variables)
print(f"Frozen layers: {len(base_model.layers)} | Params frozen: {base_model.count_params():,}")

# ── Add custom classification head ────────────────────────────────────────
inputs  = tf.keras.Input(shape=(96, 96, 3))
x       = base_model(inputs, training=False)           # frozen base
x       = tf.keras.layers.GlobalAveragePooling2D()(x)  # flatten feature maps
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)  # binary output

tl_model = tf.keras.Model(inputs, outputs, name="transfer_learning_model")
tl_model.summary()

# ── Phase 1: Train only the head (fast) ────────────────────────────────────
tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nPhase 1: Training classification head only...")
tl_model.fit(
    X_train_resized, y_train_cd,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# ── Phase 2: Fine-tune — unfreeze last 20 layers of base model ─────────────
base_model.trainable = True
# Freeze all but last 20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

print(f"\nPhase 2: Fine-tuning last 20 base layers...")
print(f"Trainable params after unfreeze: {tl_model.count_params():,}")

# Lower learning rate for fine-tuning — large lr would destroy pre-trained features
tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # 100× smaller than phase 1
    loss='binary_crossentropy',
    metrics=['accuracy']
)

tl_model.fit(
    X_train_resized, y_train_cd,
    epochs=3,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

test_loss, test_acc = tl_model.evaluate(X_test_resized, y_test_cd, verbose=0)
print(f"\nFinal Transfer Learning Accuracy: {test_acc:.2%}")


---
## Chapter 18: NLP in TensorFlow — Text Vectorization & Embeddings

Before feeding text into a neural network, you need to convert it to numbers.
TensorFlow provides `TextVectorization` — a preprocessing layer that handles the full pipeline:
tokenize → build vocabulary → encode as integers or TF-IDF.

This chapter covers the TF-native NLP pipeline. (For production NLP you'd use HuggingFace tokenizers — Chapter 15 in the companion guide.)


In [ ]:
# ── Text Vectorization Layer ──────────────────────────────────────────────
# Use case: classify product reviews as positive/negative

sample_reviews = [
    "this product is amazing and works perfectly",
    "terrible quality broke after one day",
    "great value for the price highly recommend",
    "waste of money do not buy",
    "excellent product fast shipping love it",
    "completely useless and poorly made",
    "good quality happy with purchase",
    "awful experience returning immediately",
]
labels = [1, 0, 1, 0, 1, 0, 1, 0]  # 1=positive, 0=negative

# ── Create TextVectorization layer ────────────────────────────────────────
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=100,             # vocabulary size
    output_mode='int',          # output integer token IDs
    output_sequence_length=10   # pad/truncate to 10 tokens
)

# Adapt: scan the data to build the vocabulary
vectorizer.adapt(sample_reviews)

print("Vocabulary (first 15 words):")
print(vectorizer.get_vocabulary()[:15])

# Vectorize some text
sample = ["amazing product works great"]
print("\nEncoded:", vectorizer(sample).numpy())

# ── Build a text classification model ─────────────────────────────────────
text_model = tf.keras.Sequential([
    vectorizer,                                    # text → integers

    # Embedding: each token ID → learned dense vector (learned W_e)
    tf.keras.layers.Embedding(
        input_dim=100,    # vocab size
        output_dim=16,    # embedding dimension
        name="token_embedding"
    ),

    # Global average pooling: average all token embeddings into one vector
    # Simple but effective for short texts
    tf.keras.layers.GlobalAveragePooling1D(),

    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
], name="text_classifier")

text_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

text_model.summary()

# Convert to arrays and train
X_text = np.array(sample_reviews)
y_text = np.array(labels, dtype='float32')

text_model.fit(X_text, y_text, epochs=30, verbose=0)

# Test predictions
test_reviews = [
    "absolutely love this product best purchase ever",
    "broken on arrival terrible customer service"
]
probs = text_model.predict(test_reviews, verbose=0)
for review, prob in zip(test_reviews, probs):
    print(f"\nReview: '{review[:45]}...'")
    print(f"  → {'POSITIVE' if prob[0] > 0.5 else 'NEGATIVE'} ({prob[0]:.2%})")

# ── Inspect learned embeddings ─────────────────────────────────────────────
embed_weights = text_model.get_layer("token_embedding").get_weights()[0]
vocab         = vectorizer.get_vocabulary()
print("\nLearned embedding for 'amazing':",
      embed_weights[vocab.index('amazing')] if 'amazing' in vocab else "not in vocab")
print("Embedding shape:", embed_weights.shape)  # (100, 16)
print("This is W_e — exactly the same concept as in GPT/BERT!")


---
## Chapter 19: Building a Transformer Block from Scratch
### 🟡 GPU RECOMMENDED for larger models

This chapter ties everything together by building a **complete Transformer encoder block** in TensorFlow — using the custom layer pattern from Chapter 16.

This is the exact architecture inside BERT, and every encoder inside GPT-4.

**One Transformer Block contains:**
1. Multi-Head Self-Attention (+ residual + LayerNorm)
2. Feed-Forward Network (+ residual + LayerNorm)

**Java analogy:** The Transformer block is like a design pattern — one composable unit repeated N times. GPT-3 stacks 96 of these.


In [ ]:
# ── Multi-Head Attention Layer ────────────────────────────────────────────

class MultiHeadAttention(tf.keras.layers.Layer):
    """
    Full Multi-Head Attention:
      - Splits d_model into h heads
      - Each head does scaled dot-product attention independently
      - Concatenates results and applies output projection W_O
    """

    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads  # dimension per head

        # W_Q, W_K, W_V, W_O — the four weight matrices of multi-head attention
        self.W_Q = tf.keras.layers.Dense(d_model, use_bias=False, name="W_Q")
        self.W_K = tf.keras.layers.Dense(d_model, use_bias=False, name="W_K")
        self.W_V = tf.keras.layers.Dense(d_model, use_bias=False, name="W_V")
        self.W_O = tf.keras.layers.Dense(d_model, use_bias=False, name="W_O")

    def split_heads(self, x, batch_size):
        """Reshape (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)"""
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.d_k))
        return tf.transpose(x, perm=[0, 2, 1, 3])  # (batch, heads, seq, d_k)

    def call(self, query, key, value, mask=None):
        batch_size = tf.shape(query)[0]

        # Project through W_Q, W_K, W_V
        Q = self.split_heads(self.W_Q(query), batch_size)  # (batch, heads, seq, d_k)
        K = self.split_heads(self.W_K(key),   batch_size)
        V = self.split_heads(self.W_V(value), batch_size)

        # Scaled dot-product attention per head
        scale  = tf.math.sqrt(tf.cast(self.d_k, tf.float32))
        scores = tf.linalg.matmul(Q, K, transpose_b=True) / scale  # (batch, heads, seq, seq)

        if mask is not None:
            scores += (mask * -1e9)

        weights = tf.nn.softmax(scores, axis=-1)  # attention weights
        context = tf.linalg.matmul(weights, V)    # (batch, heads, seq, d_k)

        # Concatenate heads: (batch, heads, seq, d_k) → (batch, seq, d_model)
        context = tf.transpose(context, perm=[0, 2, 1, 3])
        context = tf.reshape(context, (batch_size, -1, self.d_model))

        # Apply output projection W_O
        return self.W_O(context)


# ── Feed-Forward Network ───────────────────────────────────────────────────

class FeedForwardNetwork(tf.keras.layers.Layer):
    """
    Two-layer MLP: expand (4×) → ReLU → shrink
    Applied independently to each token position.
    One FFN per block, shared across all attention heads.
    """

    def __init__(self, d_model, d_ff, **kwargs):
        super().__init__(**kwargs)
        self.W_FF1 = tf.keras.layers.Dense(d_ff,    activation='relu', name="W_FF1")
        self.W_FF2 = tf.keras.layers.Dense(d_model, name="W_FF2")

    def call(self, x):
        return self.W_FF2(self.W_FF1(x))


# ── Complete Transformer Encoder Block ────────────────────────────────────

class TransformerEncoderBlock(tf.keras.layers.Layer):
    """
    One complete Transformer encoder block:
      1. Multi-Head Self-Attention + residual + LayerNorm
      2. Feed-Forward Network + residual + LayerNorm
    """

    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention  = MultiHeadAttention(d_model, num_heads)
        self.ffn        = FeedForwardNetwork(d_model, d_ff)
        self.norm1      = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2      = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1   = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2   = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False, mask=None):
        # ── Sub-layer 1: Self-Attention + residual + norm ──
        attn_out = self.attention(x, x, x, mask=mask)       # Q=K=V=x (self-attention)
        attn_out = self.dropout1(attn_out, training=training)
        x        = self.norm1(x + attn_out)                 # Add & Norm (residual!)

        # ── Sub-layer 2: FFN + residual + norm ────────────
        ffn_out  = self.ffn(x)
        ffn_out  = self.dropout2(ffn_out, training=training)
        x        = self.norm2(x + ffn_out)                  # Add & Norm (residual!)
        return x


# ── Build a small Transformer for sequence classification ─────────────────

d_model   = 64
num_heads = 4
d_ff      = 256
seq_len   = 20
vocab_size = 200

inputs    = tf.keras.Input(shape=(seq_len,), dtype=tf.int32)
x         = tf.keras.layers.Embedding(vocab_size, d_model)(inputs)  # W_e

# Stack 2 Transformer encoder blocks (like a tiny BERT)
x = TransformerEncoderBlock(d_model, num_heads, d_ff, name="block_1")(x)
x = TransformerEncoderBlock(d_model, num_heads, d_ff, name="block_2")(x)

# Pool: take mean of all token representations (like BERT's CLS pooling simplified)
x       = tf.keras.layers.GlobalAveragePooling1D()(x)
x       = tf.keras.layers.Dropout(0.1)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

transformer = tf.keras.Model(inputs, outputs, name="mini_transformer")
transformer.summary()

# Quick test forward pass
dummy_input = tf.random.uniform((4, seq_len), minval=0, maxval=vocab_size, dtype=tf.int32)
dummy_output = transformer(dummy_input)
print("\nMini Transformer forward pass:")
print("  Input:", dummy_input.shape)   # (4, 20)
print("  Output:", dummy_output.shape) # (4, 1) — binary classification
print("\nCongratulations — you just built a Transformer from scratch in TensorFlow!")


---
## Chapter 20: Production Tips — `@tf.function`, Mixed Precision & Profiling

When moving from experimentation to production, these techniques significantly speed up inference and reduce memory usage.

| Technique | What it does | Speedup |
|---|---|---|
| `@tf.function` | Compile Python function to TF graph (runs outside Python interpreter) | 2–10× |
| Mixed Precision | Use `float16` for compute, `float32` for weights | 2–3× on GPU |
| `tf.data.prefetch` | Overlap data loading with GPU compute | Removes I/O bottleneck |
| XLA compilation | Fuse operations at compiler level | 10–30% additional |


In [ ]:
# ── @tf.function — Graph Mode Compilation ─────────────────────────────────
import time

# Eager mode (default): Python runs line by line, like normal code
def eager_matmul(a, b):
    return tf.linalg.matmul(a, b)

# Graph mode: @tf.function traces the function once → compiles to TF graph
# Subsequent calls skip Python overhead → much faster for repeated calls
@tf.function
def graph_matmul(a, b):
    return tf.linalg.matmul(a, b)

# Warmup (first call traces the graph)
a = tf.random.normal([1000, 1000])
b = tf.random.normal([1000, 1000])
_ = graph_matmul(a, b)

# Benchmark
N = 100

start = time.perf_counter()
for _ in range(N):
    eager_matmul(a, b)
eager_time = time.perf_counter() - start

start = time.perf_counter()
for _ in range(N):
    graph_matmul(a, b)
graph_time = time.perf_counter() - start

print(f"Eager mode:  {eager_time*1000:.1f}ms for {N} calls")
print(f"Graph mode:  {graph_time*1000:.1f}ms for {N} calls")
print(f"Speedup:     {eager_time/graph_time:.1f}×")

# ── Mixed Precision Training (GPU only) ───────────────────────────────────
# Uses float16 for computation (faster on GPU) but float32 for weight storage
# Only beneficial on GPU with Tensor Cores (T4, V100, A100)
print("\nMixed precision (GPU only):")
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("  Policy set to mixed_float16")
    # Reset to float32 for rest of notebook
    tf.keras.mixed_precision.set_global_policy('float32')
    print("  Reset to float32")
except:
    print("  (Requires GPU with Tensor Cores)")

# ── Model Export for Serving ───────────────────────────────────────────────
# Wrap model in a serving function that accepts raw input (e.g. a string or bytes)
@tf.function(input_signature=[tf.TensorSpec(shape=[None, 784], dtype=tf.float32)])
def serve(inputs):
    """Production serving function — wraps model.predict() for TF Serving"""
    return {'predictions': mnist_model(inputs, training=False)}

# Export with the serving signature
mnist_model.export('mnist_production_model',
                    signatures={'serving_default': serve})
print("\nModel exported with custom serving signature")
print("Deploy with: docker run tensorflow/serving --model_base_path=/models/mnist")

# ── Summary: The Production Checklist ─────────────────────────────────────
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PRODUCTION CHECKLIST
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Use @tf.function on training/inference steps
✅ Use tf.data with .prefetch(AUTOTUNE)
✅ Validate input shapes in serving functions
✅ Save as SavedModel (not .h5) for deployment
✅ Use Mixed Precision if GPU has Tensor Cores
✅ Clip gradients (clip_by_global_norm) for stability
✅ Use EarlyStopping + ModelCheckpoint during training
✅ Monitor val_loss — not just train_loss
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")


---
## What You've Learned — Complete Roadmap

| Chapter | Topic | Key Concept |
|---|---|---|
| 1 | Tensors | Multi-dim arrays with GPU support |
| 2 | Operations | Element-wise math, matmul, reshape, broadcast |
| 3 | Variables | Mutable tensors = model weights |
| 4 | GradientTape | Automatic differentiation — the training engine |
| 5 | Sequential API | Stack layers like Lego blocks |
| 6 | Functional API | Multi-input, skip connections, residuals |
| 7 | Subclassing API | Full OOP model definition |
| 8 | Loss & Optimizers | MSE, CrossEntropy, Adam, AdamW, LR schedules |
| 9 | model.fit() | Full training + MNIST classification |
| 10 | Manual training loop | GradientTape + optimizer.apply_gradients |
| 11 | tf.data | Shuffle, batch, prefetch, map pipelines |
| 12 | Callbacks | EarlyStopping, ModelCheckpoint, custom |
| 13 | Save & Load | .keras, SavedModel, weights-only |
| 14 | CNNs | Conv2D, MaxPooling, image classification |
| 15 | RNNs / LSTMs | Sequence models, IMDB sentiment |
| 16 | Custom Layers | build(), call(), attention from scratch |
| 17 | Transfer Learning | Freeze, fine-tune, MobileNetV2 |
| 18 | NLP | TextVectorization, Embedding, text classification |
| 19 | Transformer Block | Multi-Head Attention + FFN built from scratch |
| 20 | Production | @tf.function, mixed precision, serving export |

---

## Next Steps

1. **Practice:** Re-implement each chapter with a different dataset
2. **HuggingFace:** Load pre-trained BERT/GPT models using `transformers` library
3. **Kaggle:** Enter a competition using your TF skills
4. **Projects:** Build a sentiment analyzer, image classifier, or text generator end-to-end

> *Every weight in every neural network you will ever use — GPT, BERT, LLaMA — runs on exactly these same primitives: tensors, matrix multiply, GradientTape, and gradient descent.*
